v61 
- **Verified all metric calculation with Excel** 
- Updated core.analyzer
- Replaced the `Result` pattern with exceptions and flattened the logic
- Refactored the `AlphaEngine` into a streamlined Orchestrator pattern
-  **Strict Date Logic:** No more "Time Travel" bugs.
-  **Dataclass Contracts:** No more "Magic String" typos or blind dictionaries.
-  **Exception Flow:** The `run` method is now a clean, readable story instead of a maze of `if/else` checks.
-  **Performance Workers:** Math is separated from orchestration.
- Ret_1d, explicitly turns division-by-zero results (`inf`) into `NaN`, and replace [np.inf, -np.inf] with np.nan



v60  
- Converted code from notebook to modular system.
- Fixed divide by zero warning from calculate_gain
- Added subtitle to subplots
- Added Volatility Regime plot


v59  
- Removed "nest" of if-statements in **AlphaEngine.run**
- Use **Result Pattern** to handle errors
- Change verify_analyzer_short and verify_analyzer_long gain calculation from simple return to logarithmic return
- Change calculate_gain from simple return to logarithmic return
- Remove bfill from calculate_gain to prevent backfill with future data
- Verify macro_df calculation


In [58]:
# 1. Enable Autoreload
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

def add_project_root_to_path():
    """Find notebooks_RLVR and add to sys.path."""
    current = Path.cwd()

    # Search upward for notebooks_RLVR folder
    for path in [current] + list(current.parents):
        if path.name == "notebooks_RLVR":
            sys.path.insert(0, str(path))
            print(f"✓ Added to path: {path}")
            return path
        # Also check if notebooks_RLVR exists as child (for running from stocks/)
        candidate = path / "notebooks_RLVR"
        if candidate.exists():
            sys.path.insert(0, str(candidate))
            print(f"✓ Added to path: {candidate}")
            return candidate

    raise RuntimeError("Could not find notebooks_RLVR directory")


# Run once at notebook start
add_project_root_to_path()


# 2. Force reload cached modules (run this to refresh code changes)
modules_to_reload = [
    "core.analyzer",
    "core.auditor",
    "core.contracts",    
    "core.engine",
    "core.features",
    "core.paths",
    "core.performance",
    "core.quant",
    "core.result",
    "core.settings",
    "core.utils",
    "strategy.registry",
]

for mod in modules_to_reload:
    if mod in sys.modules:
        del sys.modules[mod]


# 3. Standard imports
import pandas as pd
import numpy as np

from IPython.display import display


# 4. Fresh imports (these will re-import from disk due to cache clearing above)
from core.quant import QuantUtils
from core.analyzer import create_walk_forward_analyzer
from core.auditor import SystemAuditor as SA
from core.contracts import FilterPack, EngineInput
from core.engine import AlphaEngine
from core.features import generate_features
from core.paths import OUTPUT_DIR
from core.settings import GLOBAL_SETTINGS
from core.utils import SystemUtils as SU
from strategy.registry import METRIC_REGISTRY


# 5. Pandas display settings
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)
pd.set_option("display.max_colwidth", 50)
pd.set_option("display.precision", 4)


# 6. Load data path from .env
DATA_PATH_OHLCV = SU.load_env_from_root("DATA_PATH_OHLCV")
DATA_PATH_INDICES = SU.load_env_from_root("DATA_PATH_INDICES")
print(f"DATA_PATH_OHLCV: {DATA_PATH_OHLCV}")
print(f"DATA_PATH_INDICES: {DATA_PATH_INDICES}")

# # 7. Instantiate engine (customize DataFrames as needed)
# master_engine = AlphaEngine(
#     df_ohlcv=df_ohlcv,
#     features_df=features_df,
#     macro_df=macro_df,
#     df_close_wide=df_close_wide,
#     df_atrp_wide=df_atrp_wide,
#     df_trp_wide=df_trp_wide,
# )

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✓ Added to path: c:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR
NOTEBOOKS_RLVR_ROOT: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR

OUTPUT_DIR: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output

DATA_PATH_OHLCV: c:\Users\ping\Files_win10\python\py311\stocks\data\df_OHLCV_stocks_etfs.parquet
DATA_PATH_INDICES: c:\Users\ping\Files_win10\python\py311\stocks\data\df_indices.parquet


In [2]:
df_indices = pd.read_parquet(DATA_PATH_INDICES, engine="pyarrow")
print(f"df_indices:|n{df_indices}")

df_indices:|n                   Adj Open  Adj High  Adj Low  Adj Close               Volume
Ticker Date                                                                   
^AXJO  1992-11-22   1455.00   1455.00  1455.00    1455.00                    0
       1992-11-23   1458.40   1458.40  1458.40    1458.40                    0
       1992-11-24   1467.90   1467.90  1467.90    1467.90                    0
       1992-11-25   1459.00   1459.00  1459.00    1459.00                    0
       1992-11-26   1458.90   1458.90  1458.90    1458.90                    0
...                     ...       ...      ...        ...                  ...
^VIX3M 2026-03-09     28.04     29.09    24.86      25.34                    0
       2026-03-10     24.94     25.84    23.54      25.51                    0
       2026-03-11     25.29     25.98    24.93      24.97                    0
       2026-03-12     26.27     26.99    25.80      26.95                    0
       2026-03-13      0.00      0.00  

In [ ]:
_indices = df_indices.index.get_level_values(0).unique().tolist()
display(_indices)
df_indices.info()

In [3]:
print(f"Takes about 1.5 minutes")
df_ohlcv = pd.read_parquet(DATA_PATH_OHLCV, engine="pyarrow")

Takes about 1.5 minutes


In [ ]:
# print(f"df_ohlcv.head():\n {df_ohlcv.head()}\n")
df_ohlcv.info()

In [4]:
print(f"Takes about 3 minutes to generate_features")

features_df, macro_df = generate_features(
    df_ohlcv=df_ohlcv,
    df_indices=df_indices,
    benchmark_ticker=GLOBAL_SETTINGS["benchmark_ticker"],
)

Takes about 3 minutes to generate_features
⚡ Generating Decoupled Features (Benchmark: SPY)...


In [ ]:
print(f"features_df.info():\n{features_df.info()}\n")
print(f"features_df.index.names:\n{features_df.index.names}\n")
print(f"macro_df.info():\n{macro_df.info()}\n")
print(f"macro_df.index.names:\n{macro_df.index.names}\n")

In [ ]:
# Pre-flight Automated Audit Suite
checks = [
    SA.verify_math_integrity(),
    SA.verify_ranking_integrity(),
    SA.verify_vol_alignment_integrity(),
    SA.verify_feature_engineering_integrity(),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break

print("=" * 85)
# Separate verify_marco_engine output from intertwine with other outputs

checks = [
    SA.verify_macro_engine(
        df_ohlcv=df_ohlcv,
        df_indices=df_indices,
        original_macro_df=macro_df,
        settings=GLOBAL_SETTINGS,
    ),
]

for check in checks:
    icon = "✅" if check.ok else "🔥"
    print(f"{icon} {check.msg}")
    if not check.ok:
        print("🛑 Critical Failure. System Halted.")
        break

In [5]:
print(
    "🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)"
)

# 1. Price Matrix
df_close_wide = df_ohlcv["Adj Close"].unstack(level=0)

# 2. Volatility Matrices (Unstack and Align)
# Using features_df (the first item from the tuple)
print("   - Unstacking ATRP...")
df_atrp_wide = features_df["ATRP"].unstack(level=0).reindex_like(df_close_wide)

print("   - Unstacking TRP...")
df_trp_wide = features_df["TRP"].unstack(level=0).reindex_like(df_close_wide)

# 3. Handle Data Gaps (Sanitize the Wide Matrices)
if GLOBAL_SETTINGS["handle_zeros_as_nan"]:
    df_close_wide = df_close_wide.replace(0, np.nan)

# Forward fill up to the limit, then fill remaining with the "Disaster Detection" value
df_close_wide = df_close_wide.ffill(limit=GLOBAL_SETTINGS["max_data_gap_ffill"])
df_close_wide = df_close_wide.fillna(GLOBAL_SETTINGS["nan_price_replacement"])

print(
    f"✅ Pre-computation Complete. Tickers: {len(df_close_wide.columns)}, Days: {len(df_close_wide)}"
)
print("   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.")

🚀 Generating Wide Matrices for Instant Backtesting... (takes about 1 minute to run)
   - Unstacking ATRP...
   - Unstacking TRP...
✅ Pre-computation Complete. Tickers: 1583, Days: 16157
   Ready: df_close_wide, df_atrp_wide, df_trp_wide, and macro_df.


In [60]:
# 6. Instantiate engine (customize DataFrames as needed)
master_engine = AlphaEngine(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    macro_df=macro_df,
    df_close_wide=df_close_wide,
    df_atrp_wide=df_atrp_wide,
    df_trp_wide=df_trp_wide,
)

In [47]:
decision_dt = pd.Timestamp("2025-12-10")  # Pick a known date
lookback = 10

# A. Run via current UI logic (Manual)
# (Imagine you selected 'Sharpe (ATRP)' in the dropdown)
ui_result = master_engine.run(
    EngineInput(
        mode="Rank",
        start_date=decision_dt,
        lookback_period=lookback,
        holding_period=5,
        metric="Sharpe (ATRP)",
        benchmark_ticker="SPY",
    )
)
ui_scores = ui_result.results_df["Strategy Value"]

# B. Run via new Headless Scorer
alpha_matrix = master_engine.compute_alpha_matrix(decision_dt, lookback)
headless_scores = alpha_matrix["Sharpe (ATRP)"].reindex(ui_scores.index)

# VERIFICATION
pd.testing.assert_series_equal(ui_scores, headless_scores, check_names=False)
print("✅ Step 1 Success: Headless Scorer matches UI logic perfectly.")

DEBUG: 937 stocks passed filters on 2025-12-10
✅ Step 1 Success: Headless Scorer matches UI logic perfectly.


In [48]:
# --- TEST 2: NORMALIZATION INTEGRITY ---
raw_matrix = master_engine.compute_alpha_matrix(decision_dt, lookback)
norm_matrix = master_engine.normalize_alpha_matrix(raw_matrix)

# Verification A: Mean should be near 0
mean_check = norm_matrix.mean().abs().max()
# Verification B: Std should be near 1
std_check = norm_matrix.std().max()

print(f"Max Mean Offset: {mean_check:.6f} (Target: < 0.01)")
print(f"Max Std: {std_check:.6f} (Target: ~ 1.0)")

if mean_check < 0.01 and 0.8 < std_check < 1.2:
    print("✅ Step 2 Success: The Alpha Matrix is now 'Agent-Ready'.")
else:
    print("❌ Step 2 Failed: Normalization drift detected.")

Max Mean Offset: 0.009404 (Target: < 0.01)
Max Std: 1.000000 (Target: ~ 1.0)
✅ Step 2 Success: The Alpha Matrix is now 'Agent-Ready'.


In [49]:
# --- TEST 2.1: REGIME AWARENESS VERIFICATION ---
# decision_dt = pd.Timestamp("2025-12-10")
context = master_engine.compute_context_vector(decision_dt)

print("--- Agent Macro Context ---")
print(context)

# Check against your Plotly Image (Dec 10 values):
# Trend (Green line): Should be ~10% (0.10)
# Trend Vel (Orange line): Should be slightly below 0.0
# VIX Ratio: Chart says 0.86
# VIX Z (Purple line): Should be around -1.0

print(f"\nVerification Check:")
print(f"VIX Ratio in Context: {context['Context_Vix_Ratio'] + 1:.2f} (Target: 0.86)")

--- Agent Macro Context ---
Context_Trend        1.1401
Context_Vel_Z       -0.3619
Context_Vix_Z       -0.8380
Context_Vix_Ratio   -0.1850
dtype: float64

Verification Check:
VIX Ratio in Context: 0.81 (Target: 0.86)


In [61]:
# --- TEST 3: DISCOVERY REPLICATION ---
# Let's see if the agent can replicate "Sharpe (ATRP)" by weighting it 1.0
# and all others 0.0.

# 1. Identify index of "Sharpe (ATRP)" in the registry
registry_keys = list(METRIC_REGISTRY.keys())
sharpe_atrp_idx = registry_keys.index("Sharpe (ATRP)")

# 2. Create the weight vector (1.0 for Sharpe ATRP, 0.0 for others)
action_weights = np.zeros(len(registry_keys))
action_weights[sharpe_atrp_idx] = 1.0

# 3. Run Discovery
reward = master_engine.run_discovery_action(
    decision_date=pd.Timestamp("2025-12-10"),
    lookback_period=21,
    holding_period=5,
    weights=action_weights,
)

print(f"Discovery Reward (Weighted): {reward:.4f}")

# 4. Compare to UI result
# Run your master_engine.run() with "Sharpe (ATRP)" for the same date
ui_result = master_engine.run(
    EngineInput(
        mode="Ranking",
        start_date=pd.Timestamp("2025-12-10"),
        lookback_period=21,
        holding_period=5,
        metric="Sharpe (ATRP)",
        benchmark_ticker="SPY",
    )
)
ui_reward = ui_result.perf_metrics["holding_p_sharpe"]

print(f"Original UI Reward: {ui_reward:.4f}")

Discovery Reward (Weighted): 28.2954
DEBUG: 937 stocks passed filters on 2025-12-10
Original UI Reward: 28.2954


In [35]:
macro_df.loc[decision_dt]

Mkt_Ret              0.0066
Macro_Trend          0.1140
Macro_Trend_Vel     -0.0047
Macro_Trend_Vel_Z   -0.3619
Macro_Trend_Mom     -0.0047
Macro_Vix_Z         -0.8380
Macro_Vix_Ratio      0.8150
Name: 2025-12-10 00:00:00, dtype: float64

In [7]:
# universe_subset=None means "Scan the whole market"
analyzer1, stage1_pack = create_walk_forward_analyzer(
    master_engine, universe_subset=None
)

print("🚀 Ready for Stage 1: Run Simulation for first filter.")
analyzer1.show()

🚀 Ready for Stage 1: Run Simulation for first filter.
DEBUG: 937 stocks passed filters on 2025-12-10


In [ ]:
# tick_price_252 = analyzer1.last_run.tickers
# tick_price_189 = analyzer1.last_run.tickers
# tick_price_126 = analyzer1.last_run.tickers
# tick_price_63 = analyzer1.last_run.tickers
# tick_price_42 = analyzer1.last_run.tickers
# tick_price_21 = analyzer1.last_run.tickers
# tick_price_10 = analyzer1.last_run.tickers

In [ ]:
# tick_sharpe_atrp_252 = analyzer1.last_run.tickers
# tick_sharpe_atrp_189 = analyzer1.last_run.tickers
# tick_sharpe_atrp_126 = analyzer1.last_run.tickers
# tick_sharpe_atrp_63 = analyzer1.last_run.tickers
# tick_sharpe_atrp_42 = analyzer1.last_run.tickers
# tick_sharpe_atrp_21 = analyzer1.last_run.tickers
# tick_sharpe_atrp_10 = analyzer1.last_run.tickers

In [ ]:
union, intersection = set_operations(
    tick_sharpe_atrp_252,
    tick_sharpe_atrp_189,
    tick_sharpe_atrp_126,
    tick_sharpe_atrp_63,
    tick_sharpe_atrp_42,
    tick_sharpe_atrp_21,
    tick_sharpe_atrp_10,
)

print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_sharpe_atrp = intersection
# int_252_189_126 = intersection

In [ ]:
print(list_to_string(int_sharpe_atrp))

In [ ]:
def list_to_string(items, separator=", ", last_separator=None):
    """
    Convert list to string with customizable separators

    Parameters:
    -----------
    items : list of strings
    separator : str, default ', '
        Separator between items
    last_separator : str, optional
        Special separator for last item (e.g., ' and ', ' or ')

    Returns:
    --------
    str : Formatted string

    Examples:
    ---------
    list_to_string(['a', 'b', 'c'])              # "a, b, c"
    list_to_string(['a', 'b', 'c'], ' | ')        # "a | b | c"
    list_to_string(['a', 'b', 'c'], ', ', ' and ')  # "a, b and c"
    """
    if not items:
        return ""

    if len(items) == 1:
        return str(items[0])

    if last_separator and len(items) > 1:
        return separator.join(items[:-1]) + last_separator + items[-1]

    return separator.join(str(item) for item in items)


# Usage examples
print(list_to_string(["a", "b", "c"]))  # a, b, c
print(list_to_string(["a", "b", "c"], " | "))  # a | b | c
print(list_to_string(["a", "b", "c"], ", ", " and "))  # a, b and c
print(list_to_string(["a", "b", "c"], ", ", ", and "))  # a, b, and c
print(list_to_string(["apple", "banana"], ", ", " and "))  # apple and banana

In [ ]:
union, intersection = set_operations(
    tick_price_252,
    tick_price_189,
    tick_price_126,
    tick_price_63,
    tick_price_42,
    tick_price_21,
    tick_price_10,
)

print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_price = intersection
# int_252_189_126 = intersection

In [ ]:
union, intersection = set_operations(
    int_sharpe_atrp,
    int_price,
)
print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

int_shp_atrp_price = intersection

In [ ]:
def set_operations(*lists):
    """
    Find sorted union, intersection, and elements unique to first list

    Parameters:
    -----------
    *lists : variable number of lists of strings

    Returns:
    --------
    tuple : (sorted_union, sorted_intersection, unique_to_first_list)
        - union: all unique elements from all lists
        - intersection: common elements in ALL lists
        - unique_to_first: elements only in first list, not in any other list

    Examples:
    ---------
    union, intersection, unique_first = set_operations(list1, list2)
    union, intersection, unique_first = set_operations(list1, list2, list3, list4)
    """

    if not lists:
        return [], [], []

    # Convert each list to a set
    sets = [set(lst) for lst in lists]
    first_set = sets[0]
    other_sets = sets[1:] if len(sets) > 1 else []

    # Union: all unique elements from all lists
    union_set = set().union(*sets)
    union = sorted(union_set)

    # Intersection: common elements in ALL lists
    intersection_set = first_set.intersection(*other_sets) if other_sets else first_set
    intersection = sorted(intersection_set)

    # Unique to first list: in first_set but NOT in any other set
    # Using difference: first_set - (union of all other sets)
    if other_sets:
        all_others = set().union(*other_sets)
        unique_to_first_set = (
            first_set - all_others
        )  # or first_set.difference(all_others)
    else:
        unique_to_first_set = (
            first_set  # If only one list, all elements are "unique" to it
        )

    unique_to_first = sorted(unique_to_first_set)

    return union, intersection, unique_to_first


#

In [ ]:
# Get decision date from last run
decision_date_last_run = FilterPack(decision_date=analyzer1.last_run.decision_date)

# 1. LAUNCH STAGE 2 (Cascade)
# universe_subset=analyzer1.last_run.tickers means "Scan the whole market"
analyzer2, stage1_pack = create_walk_forward_analyzer(
    master_engine,
    universe_subset=analyzer1.last_run.tickers,
    # universe_subset=None,
    filter_pack=decision_date_last_run,
)

print("🚀 Ready for Stage 2: Run Simulation for 2nd filter.")
analyzer2.show()

In [ ]:
###############################
###############################

In [11]:
my_analyzer = analyzer1

my_res = SU.visualize_analyzer_structure(my_analyzer)

🔍 HIGH-TRANSPARENCY AUDIT MAP
[  0] 📦 audit_pack (EngineOutput)
[  1]   📈 portfolio_series (shape=(17,))
[  2]   📈 benchmark_series (shape=(17,))
[  3]   🧮 normalized_plot_data (shape=(17, 10))
[  4]   📂 tickers (len=10)
[  5]     📄 index_0 (str)
[  6]     📄 index_1 (str)
[  7]     📄 index_2 (str)
[  8]     📄 index_3 (str)
[  9]     📄 index_4 (str)
[ 10]     📄 index_5 (str)
[ 11]     📄 index_6 (str)
[ 12]     📄 index_7 (str)
[ 13]     📄 index_8 (str)
[ 14]     📄 index_9 (str)
[ 15]   📈 initial_weights (shape=(10,))
[ 16]   📂 perf_metrics (len=24)
[ 17]     🔢 full_p_gain (float)
[ 18]     🔢 full_p_sharpe (float)
[ 19]     🔢 full_p_sharpe_atrp (float)
[ 20]     🔢 full_p_sharpe_trp (float)
[ 21]     🔢 lookback_p_gain (float)
[ 22]     🔢 lookback_p_sharpe (float)
[ 23]     🔢 lookback_p_sharpe_atrp (float)
[ 24]     🔢 lookback_p_sharpe_trp (float)
[ 25]     🔢 holding_p_gain (float)
[ 26]     🔢 holding_p_sharpe (float)
[ 27]     🔢 holding_p_sharpe_atrp (float)
[ 28]     🔢 holding_p_sharpe_tr

In [ ]:
def set_operations(list1, list2):
    """
    Find sorted union and intersection of two lists of strings

    Parameters:
    -----------
    list1, list2 : list of strings

    Returns:
    --------
    tuple : (sorted_union, sorted_intersection)
    """

    # Convert to sets for operations
    set1 = set(list1)
    set2 = set(list2)

    # Union: all unique elements from both lists
    union = sorted(set1 | set2)  # or set1.union(set2)

    # Intersection: common elements in both lists
    intersection = sorted(set1 & set2)  # or set1.intersection(set2)

    return union, intersection


# Example usage
list_a = ["apple", "banana", "cherry", "date", "elderberry"]
list_b = ["banana", "date", "fig", "grape", "apple"]

union, intersection = set_operations(list_a, list_b)

print(f"List 1: {list_a}")
print(f"List 2: {list_b}")
print(f"\nUnion ({len(union)} items): {union}")
print(f"Intersection ({len(intersection)} items): {intersection}")

In [ ]:
union, intersection = set_operations(list_a, list_b)

In [ ]:
tick_sharpe_trp_252 = SU.peek(4, my_res)

In [ ]:
analyzer1.last_run.tickers
analyzer1.last_run.start_date
analyzer1.last_run.holding_end_date

In [ ]:
# 3. Post-flight Reconciliation
audit = SA.verify_analyzer_short(my_analyzer)
if not audit.ok:
    print(f"🚨 ALERT: {audit.msg}")
    # You could trigger a notification or log the failure here

In [ ]:
audit = SA.verify_analyzer_long(my_analyzer, n_tickers=5)

In [ ]:
# Takes 4 seconds to run, checks selected tickers from analyzer1
SA.audit_feature_engineering_integrity(my_analyzer, mode="last_run")

### Audit Every Tickers in features_df, Takes 3 minutes 

In [ ]:
# # Takes 3 minutes to run, checks every tickers ~1580 tickers
# SA.audit_feature_engineering_integrity(my_analyzer, mode="system")

### Export Ticker's OHLCV and Features

In [12]:
f_name_excel = OUTPUT_DIR / "Audit_Verification_Report.xlsx"

SU.export_audit_to_excel(audit_pack=my_analyzer.last_run, filename=f_name_excel)

📂 [EXCEL AUDIT] Building full transparency report: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx
✨ Audit Report Complete: C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR\output\Audit_Verification_Report.xlsx


WindowsPath('C:/Users/ping/Files_win10/python/py311/stocks/notebooks_RLVR/output/Audit_Verification_Report.xlsx')

In [ ]:
f_name_csv = OUTPUT_DIR / "all_tickers_data_stacked.csv"

# Single call replaces your 3 cells
file_path = SU.export_last_run_tickers_data_to_csv(
    analyzer=my_analyzer,
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    filename=f_name_csv,
)

In [ ]:
SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=my_analyzer.last_run.tickers,
    date_start=my_analyzer.last_run.start_date,
    date_end=analyzer1.last_run.holding_end_date,
    verbose=False,
)

In [ ]:
_tic = "NVDA"
_start_date = "2025-03-12"
_end_date = "2026-03-12"
print(features_df.loc[_tic][_start_date:_end_date])
# features_df.loc[_tic][_start_date:_end_date][["Ret_1d", "Consistency"]]

In [ ]:
result = SU.create_combined_dict(
    df_ohlcv=df_ohlcv,
    features_df=features_df,
    tickers=[_tic],
    date_start=_start_date,
    date_end=_end_date,
    verbose=False,
)
print(result[_tic])